# Granite 4.0-1B text generation with OpenVINO

Run the dense, all-attention Granite 4.0-1B instruction model locally with OpenVINO GenAI. Export and compress its weights with Optimum Intel and NNCF, stream chat responses, summarize a short document, and call an offline temperature-conversion tool.

Model cards: [`ibm-granite/granite-4.0-1b`](https://huggingface.co/ibm-granite/granite-4.0-1b).

This tutorial demonstrates how to:

- convert the model to OpenVINO IR with Optimum Intel and compress the weights with NNCF;
- stream chat responses and pass conversation history;
- summarize a short document;
- validate a model-generated tool call and execute an offline function;
- launch an interactive chat demo.

The checkpoint uses the `GraniteMoeHybridForCausalLM` class, but its configuration contains 40 attention layers and no Mamba layers or experts. It is not the hybrid `granite-4.0-h-1b` model. The model weights are licensed under [Apache-2.0](https://huggingface.co/ibm-granite/granite-4.0-1b/blob/main/LICENSE).

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO. It may be fully supported and validated in the future.




### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/granite-4.0/granite-4.0.ipynb" />

#### Table of contents:

- [Prerequisites](#Prerequisites)
    - [Installation](#Installation)
- [Select model and export options](#Select-model-and-export-options)
- [Select inference device](#Select-inference-device)
- [Convert the model to OpenVINO IR](#Convert-the-model-to-OpenVINO-IR)
- [Chat with streamed output](#Chat-with-streamed-output)
- [Summarize a short document](#Summarize-a-short-document)
- [Call an offline tool](#Call-an-offline-tool)
- [Interactive demo](#Interactive-demo)
- [Clean up](#Clean-up)



## Prerequisites
[back to top ⬆️](#Table-of-contents:)

### Installation
[back to top ⬆️](#Table-of-contents:)

This notebook uses unreleased fixes for an existing model, rather than a newly released checkpoint:

- Optimum Intel is pinned to commit `c9c3b841cd74728fc14952e7ef0680be416d1833` of [PR 2117](https://github.com/huggingface/optimum-intel/pull/2117), which exports a non-default activation scale to prevent FP16 overflow on GPU.
- Matching OpenVINO, Tokenizers and GenAI nightly packages from November 16, 2026 include the [GPU scaling fix](https://github.com/openvinotoolkit/openvino/pull/39233) and the [position-ID transformation](https://github.com/openvinotoolkit/openvino/pull/39112) needed by paged attention for this checkpoint.
- Transformers 5.5.0 satisfies the pinned exporter's minimum version. NNCF 3.4.0 provides weight compression.

Restart the notebook kernel after installing or changing these packages.

In [ ]:
import urllib.request
from pathlib import Path

# Helpers come from `utils` of the cloned repository, otherwise from GitHub; local copies are overwritten
helper_files = ["notebook_utils.py", "cmd_helper.py", "pip_helper.py"]
repo_utils = Path("../../utils")
in_repo = (repo_utils / "notebook_utils.py").is_file()
if in_repo:
    helper_contents = {helper_file: (repo_utils / helper_file).read_bytes() for helper_file in helper_files}
else:
    try:
        helper_contents = {}
        for helper_file in helper_files:
            url = f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{helper_file}"
            with urllib.request.urlopen(url, timeout=10) as response:
                helper_contents[helper_file] = response.read()
    except OSError as error:
        if any(not Path(helper_file).is_file() for helper_file in helper_files):
            raise
        print(f"Could not update helper files, using the existing copies: {error}")
        helper_contents = {}

for helper_file, content in helper_contents.items():
    Path(helper_file).write_bytes(content)

from notebook_utils import setup_notebook_venv

# Requirements installed below go to a notebook-specific virtual environment in the `.venv` folder
setup_notebook_venv()

In [ ]:
from pip_helper import pip_install

pip_install(
    "-q",
    "git+https://github.com/huggingface/optimum-intel.git@c9c3b841cd74728fc14952e7ef0680be416d1833",
    "nncf==3.4.0",
    "transformers==5.5.0",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
)
pip_install(
    "-q",
    "--pre",
    "openvino==2026.5.1.dev20261116",
    "openvino-tokenizers==2026.5.1.0.dev20261116",
    "openvino-genai==2026.5.1.0.dev20261116",
    "--extra-index-url",
    "https://storage.openvinotoolkit.org/simple/wheels/nightly",
)
pip_install("-q", "gradio==6.32.0", "ipywidgets", "accelerate")

import ipywidgets as widgets
from IPython.display import display

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("granite-4.0.ipynb")

## Select model and export options
[back to top ⬆️](#Table-of-contents:)

Select the weight format for the instruction-tuned 1B checkpoint. INT8 weight compression is the default; FP16 and INT4 are alternatives to compare on your device. This is weight compression, not full activation quantization. INT4 uses asymmetric groups of 128 weights with a ratio of 1.0. No calibration dataset, AWQ or scale estimation is requested.

In [ ]:
model_ids = ["ibm-granite/granite-4.0-1b"]

model_id_widget = widgets.Dropdown(options=model_ids, value=model_ids[0], description="Model:")
weight_format = widgets.Dropdown(options=["int8", "int4", "fp16"], value="int8", description="Weights:")
display(model_id_widget, weight_format)

## Select inference device
[back to top ⬆️](#Table-of-contents:)

Select CPU or an available Intel GPU. NPU is excluded because support for this exact all-attention checkpoint has not been established.

In [ ]:
from notebook_utils import device_widget

device = device_widget("CPU", exclude=["NPU", "AUTO"])
device

In [ ]:
model_id = model_id_widget.value
precision = weight_format.value
revision = "6a7381ba1f54d684ff508d991aeb7dc580157103"
model_dir = Path(model_id.split("/")[-1]) / revision[:12] / precision
print(f"Model: {model_id}, weights: {precision}, OpenVINO IR directory: {model_dir}")

## Convert the model to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

Download the pinned Hugging Face snapshot, then export its local path with `optimum-cli export openvino`, including the tokenizer and detokenizer needed by GenAI. The CLI does not accept a revision argument. `--weight-format` applies NNCF weight compression during export. The source revision and weight format identify separate output directories. No OpenVINO-author preconverted IR was found when this notebook was prepared.

In [ ]:
from huggingface_hub import snapshot_download

from cmd_helper import optimum_cli

additional_args = {"weight-format": precision, "task": "text-generation-with-past"}

if precision == "int4":
    additional_args.update({"group-size": "128", "ratio": "1.0"})

required_files = ["config.json", "tokenizer_config.json"]
required_files += [f"{name}.{suffix}" for name in ["openvino_model", "openvino_tokenizer", "openvino_detokenizer"] for suffix in ["xml", "bin"]]
if all((model_dir / filename).is_file() for filename in required_files):
    print(f"Using existing OpenVINO model from {model_dir.resolve()}")
else:
    source_dir = snapshot_download(repo_id=model_id, revision=revision, allow_patterns=["*.json", "*.jinja", "*.safetensors", "*.model", "*.txt"])
    optimum_cli(source_dir, model_dir, additional_args=additional_args)

## Chat with streamed output
[back to top ⬆️](#Table-of-contents:)

Load one GenAI pipeline and reuse it for all examples. The Hugging Face tokenizer renders Granite's chat template, including tool definitions when provided. GenAI receives an already formatted prompt, so its own template application is disabled. Each call supplies its complete conversation explicitly; separate Gradio sessions do not share chat history. Input and output token limits bound the examples.

In [ ]:
from collections.abc import Callable
from functools import partial

import openvino_genai as ov_genai
from transformers import AutoTokenizer, PreTrainedTokenizerBase

MAX_INPUT_TOKENS = 2048
MAX_NEW_TOKENS = 512

tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=False)
pipe = ov_genai.LLMPipeline(str(model_dir), device.value, CACHE_DIR=str(model_dir / "model_cache"))


def generate_chat(
    messages: list[dict[str, object]],
    pipe: ov_genai.LLMPipeline,
    tokenizer: PreTrainedTokenizerBase,
    max_input_tokens: int,
    max_new_tokens: int,
    tools: list[dict[str, object]] | None = None,
    streamer: Callable[[str], ov_genai.StreamingStatus] | None = None,
) -> str:
    prompt = tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True)
    input_tokens = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    if len(input_tokens) > max_input_tokens:
        raise ValueError(f"The conversation has {len(input_tokens)} tokens; shorten it to at most {max_input_tokens}.")
    config = ov_genai.GenerationConfig()
    config.max_new_tokens = max_new_tokens
    config.do_sample = False
    config.apply_chat_template = False
    return pipe.generate(prompt, generation_config=config, streamer=streamer)


chat = partial(generate_chat, pipe=pipe, tokenizer=tokenizer, max_input_tokens=MAX_INPUT_TOKENS, max_new_tokens=MAX_NEW_TOKENS)


def print_stream(subword: str) -> ov_genai.StreamingStatus:
    print(subword, end="", flush=True)
    return ov_genai.StreamingStatus.RUNNING


messages = [{"role": "user", "content": "What is the capital of France? Answer in one sentence."}]
print("CHAT RESPONSE: ", end="")
answer = chat(messages, streamer=print_stream)
print()
messages += [{"role": "assistant", "content": answer}, {"role": "user", "content": "Which country did we just discuss?"}]
print("FOLLOW-UP RESPONSE:", chat(messages))

## Summarize a short document
[back to top ⬆️](#Table-of-contents:)

Start a fresh conversation by supplying a new message list. The text below is a fictional example, not a downloaded dataset. Ask the model to use only the supplied facts.

In [ ]:
document = """The town library will reopen on Monday after a renovation.
The ground floor now has a quiet reading room and six public computers.
A free workshop on finding reliable online information takes place on Wednesday at 3 PM.
Visitors can register for the workshop at the front desk. Book borrowing remains free."""
summary_messages = [
    {"role": "system", "content": "Summarize only the supplied document. Do not add facts."},
    {"role": "user", "content": f"Summarize this document in two sentences:\n\n{document}"},
]
print("SUMMARY RESPONSE:", chat(summary_messages))

## Call an offline tool
[back to top ⬆️](#Table-of-contents:)

Granite's template accepts function definitions through `tools`. The model proposes a JSON call inside `<tool_call>` tags; the application, not the model, decides whether to execute it. This example allows exactly one temperature conversion with bounded numeric arguments. It rejects unknown functions, extra arguments and malformed calls, and never evaluates generated Python or contacts an external service. If the model does not return a complete tool call, no function is executed.

After a valid call, append the result as a `tool` message and let the model explain it. Text similarity alone does not establish tool-calling correctness.

In [ ]:
import json
import re

tools = [
    {
        "type": "function",
        "function": {
            "name": "convert_temperature",
            "description": "Convert a temperature between Celsius and Fahrenheit.",
            "parameters": {
                "type": "object",
                "properties": {
                    "value": {"type": "number", "minimum": -1000000, "maximum": 1000000},
                    "from_unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                    "to_unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["value", "from_unit", "to_unit"],
                "additionalProperties": False,
            },
        },
    }
]


def execute_temperature_call(response: str) -> str | None:
    calls = re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", response, flags=re.DOTALL)
    if not calls:
        return None
    if len(calls) != 1:
        raise ValueError("This example accepts exactly one tool call.")
    call = json.loads(calls[0])
    if not isinstance(call, dict) or set(call) != {"name", "arguments"} or call["name"] != "convert_temperature":
        raise ValueError("Only the convert_temperature function with name and arguments is allowed.")
    arguments = call["arguments"]
    if not isinstance(arguments, dict) or set(arguments) != {"value", "from_unit", "to_unit"}:
        raise ValueError("Expected exactly value, from_unit and to_unit arguments.")
    value = arguments["value"]
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not -1000000 <= value <= 1000000:
        raise ValueError("Temperature must be a finite number between -1000000 and 1000000.")
    if arguments["from_unit"] not in ("celsius", "fahrenheit") or arguments["to_unit"] not in ("celsius", "fahrenheit"):
        raise ValueError("Temperature units must be celsius or fahrenheit.")
    celsius = value if arguments["from_unit"] == "celsius" else (value - 32) * 5 / 9
    converted = celsius if arguments["to_unit"] == "celsius" else celsius * 9 / 5 + 32
    return json.dumps({"value": round(converted, 4), "unit": arguments["to_unit"]})


tool_messages = [{"role": "user", "content": "Convert 20 degrees Celsius to Fahrenheit using the available tool."}]
tool_response = chat(tool_messages, tools=tools)
print("TOOL PROPOSAL:", tool_response)
tool_result = execute_temperature_call(tool_response)
if tool_result is None:
    print("No complete tool call was returned; no function was executed.")
else:
    print("TOOL RESULT:", tool_result)
    tool_messages += [{"role": "assistant", "content": tool_response}, {"role": "tool", "content": tool_result}]
    print("TOOL ANSWER:", chat(tool_messages, tools=tools))

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

Run the cell below to chat with the same model. Each request sends that browser session's history, with the same input/output token limits as the examples above. Requests are queued one at a time because they share one pipeline. Clear the chat when its history reaches the input limit.

The demo first launches locally. If that fails, it reports the error and retries with a publicly accessible Gradio share link.

In [ ]:
from gradio_helper import make_demo

demo = make_demo(generate=chat)

# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/
try:
    demo.launch(debug=True)
except Exception as error:
    print(f"Local Gradio launch failed: {error}. Retrying with a public share link.")
    demo.launch(debug=True, share=True)

## Clean up
[back to top ⬆️](#Table-of-contents:)

Uncomment and run the cell below to remove the selected export and its compiled cache. The Hugging Face download cache and notebook virtual environment are separate. See the [GenAI documentation](https://docs.openvino.ai/2026/openvino-workflow-generative/inference-with-genai.html) for other generation APIs and the [benchmark tools](https://github.com/openvinotoolkit/openvino.genai/tree/master/tools) for performance and output-similarity evaluation.

In [ ]:
# import shutil

# shutil.rmtree(model_dir)